# RealVisXL V4.0 Face-Lock Image-to-Image Studio
Uses safe CPU/GPU offload detection to avoid RuntimeError when accelerator is missing.

In [ ]:
!pip install -q diffusers transformers accelerate torch torchvision gradio

import torch
from diffusers import AutoPipelineForImage2Image
import gradio as gr

has_cuda = torch.cuda.is_available()
device = "cuda" if has_cuda else "cpu"
dtype = torch.float16 if has_cuda else torch.float32

print(f"Loading RealVisXL Face-Lock Pipeline on {device.upper()}...")
pipe_i2i = AutoPipelineForImage2Image.from_pretrained(
    "SG161222/RealVisXL_V4.0",
    torch_dtype=dtype,
    use_safetensors=True
)

if has_cuda:
    pipe_i2i.enable_sequential_cpu_offload()
else:
    pipe_i2i.to("cpu")
    print("WARNING: Running on CPU. Go to Runtime -> Change runtime type -> Select T4 GPU for fast generation!")

def edit(image, prompt, negative_prompt, strength, steps, cfg):
    if image is None:
        return None
    
    image = image.convert("RGB")
    image.thumbnail((1024, 1024))
    
    full_prompt = prompt + ", highly detailed, sharp focus, masterpiece"
    
    img = pipe_i2i(
        prompt=full_prompt,
        negative_prompt=negative_prompt,
        image=image,
        strength=float(strength),
        num_inference_steps=int(steps),
        guidance_scale=float(cfg)
    ).images[0]
    return img

demo = gr.Interface(
    fn=edit,
    inputs=[
        gr.Image(type="pil", label="Source Image"),
        gr.Textbox(label="Prompt (What to enhance/adjust)", value="photorealistic, high quality, sharp details"),
        gr.Textbox(label="Negative Prompt", value="blurry, low quality, deformed"),
        gr.Slider(0.1, 0.5, value=0.25, label="Strength (KEEP LOW: 0.2 - 0.3 preserves face and identity)"),
        gr.Slider(20, 50, value=30, step=1, label="Steps"),
        gr.Slider(4.0, 10.0, value=7.5, step=0.5, label="CFG Scale")
    ],
    outputs=gr.Image(label="Result (Face Preserved)"),
    title="RealVisXL Face-Lock Studio"
)
demo.launch(share=True, debug=True)